# Compute Engine 인스턴스 생성 및 Ops Agent 정책 설정 예제

이 노트북은 Google Cloud SDK(`gcloud`)를 사용하여 Compute Engine VM 인스턴스를 생성하고, Google Cloud Ops Agent 정책을 생성 및 적용합니다.

- **인스턴스 이름**: `instance-20260915-143200`
- **프로젝트 ID**: `iceu-songpa07`
- **영역 (Zone)**: `us-central1-a`
- **머신 유형**: `e2-medium` (2 vCPU, 4GB RAM)
- **디스크**: 10GB `pd-balanced`
- **OS 이미지**: `debian-13-trixie-v20260908`

## 💰 동일 사양 최저가 리전 분석 (Top 3)
동일한 스펙(`e2-medium`, 10GB `pd-balanced` 디스크, Standard 모델, Debian OS)으로 생성할 때 전 세계 GCP 리전 중 가장 저렴한 리전과 비용을 비교합니다.

In [1]:
from find_cheapest_region import print_cheapest_summary

# 최저가 리전 TOP 3 및 글로벌 리전 요금 비교표 출력
print_cheapest_summary()

💰 Google Cloud Compute Engine 리전별 요금 분석 (e2-medium + 10GB pd-balanced)
기준 스펙: e2-medium (2 vCPU, 4GB RAM), pd-balanced 10GB, Standard Provisioning, Debian OS
월 환산 기준: 730시간 (24시간 30.4일 상시 가동 기준)
-------------------------------------------------------------------------------------

🏆 [가장 저렴한 리전 TOP 3]
-------------------------------------------------------------------------------------
순위   | 리전 (Zone)              | 위치                         | 시간당 비용         | 월간 총 비용
-------------------------------------------------------------------------------------
1위   | us-central1 (us-central1-a) | 미국 아이오와 (Iowa)             | $0.0349/시간   | $25.46/월
       ㄴ [상세] VM: $24.46/월 (시간당 $0.0335) + 디스크: $1.00/월
2위   | us-east1 (us-east1-b)  | 미국 사우스캐롤라이나 (South Carolina) | $0.0349/시간   | $25.46/월
       ㄴ [상세] VM: $24.46/월 (시간당 $0.0335) + 디스크: $1.00/월
3위   | us-west1 (us-west1-b)  | 미국 오레곤 (Oregon)            | $0.0349/시간   | $25.46/월
       ㄴ [상세] VM: $24.46/월 (시간당 $0.0335) + 디스크: $1.00/월

📊 [주요 글로벌

### 🚀 최저가 리전(Top 1~3)으로 인스턴스 생성
최저가 그룹 중 원하는 리전을 선택하여 동일한 옵션으로 인스턴스를 생성합니다.
- **us-central1-a** (미국 아이오와 - 기본값, 최저가)
- **us-east1-b** (미국 사우스캐롤라이나 - 최저가)
- **us-west1-b** (미국 오레곤 - 최저가)

In [2]:
import os
import sys
import shutil
import subprocess

# gcloud 실행 파일 경로 자동 탐색 (Windows 호환성 완벽 지원)
gcloud_bin = shutil.which("gcloud.cmd") or shutil.which("gcloud") or "gcloud"
is_windows = sys.platform.startswith("win")

# 최저가 리전 설정
SELECTED_ZONE = "us-central1-a"
PROJECT_ID = "iceu-songpa07"
INSTANCE_NAME = "instance-cheapest-e2m"

create_cmd = [
    gcloud_bin, "compute", "instances", "create", INSTANCE_NAME,
    f"--project={PROJECT_ID}",
    f"--zone={SELECTED_ZONE}",
    "--machine-type=e2-medium",
    "--network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default",
    "--metadata=enable-osconfig=TRUE",
    "--maintenance-policy=MIGRATE",
    "--provisioning-model=STANDARD",
    "--service-account=1063515563667-compute@developer.gserviceaccount.com",
    "--scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append",
    f"--create-disk=auto-delete=yes,boot=yes,device-name={INSTANCE_NAME},image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced",
    "--no-shielded-secure-boot",
    "--shielded-vtpm",
    "--shielded-integrity-monitoring",
    "--labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud",
    "--reservation-affinity=any",
    "--quiet"  # 대화형 프롬프트 대기 방지
]

print(f"[*] [{SELECTED_ZONE}] 최저가 인스턴스 ({INSTANCE_NAME}) 생성 요청 시작 (약 30~50초 소요)...", flush=True)
res = subprocess.run(create_cmd, shell=is_windows, stdin=subprocess.DEVNULL, capture_output=True, text=True, encoding="utf-8", errors="replace")
if res.returncode == 0:
    print("[+] 인스턴스가 성공적으로 생성되었습니다!")
    print(res.stdout)
else:
    print("[-] 생성 중 오류 발생:", res.stderr)

[*] [us-central1-a] 최저가 인스턴스 (instance-cheapest-e2m) 생성 요청 시작 (약 30~50초 소요)...
[+] 인스턴스가 성공적으로 생성되었습니다!
NAME: instance-cheapest-e2m
ZONE: us-central1-a
MACHINE_TYPE: e2-medium
PREEMPTIBLE: 
INTERNAL_IP: 10.128.0.4
EXTERNAL_IP: 8.35.194.5
STATUS: RUNNING



### 방법 1. 원본 설정 Python 코드로 실행
기존 us-central1-a 원본 인스턴스(`instance-20260914-054908`) 및 Ops Agent 정책 생성 코드입니다.

In [3]:
import os
import sys
import shutil
import subprocess

gcloud_bin = shutil.which("gcloud.cmd") or shutil.which("gcloud") or "gcloud"
is_windows = sys.platform.startswith("win")

# 1. Compute Engine 인스턴스 생성
create_instance_cmd = [
    gcloud_bin, "compute", "instances", "create", "instance-20260914-054908",
    "--project=iceu-songpa07",
    "--zone=us-central1-a",
    "--machine-type=e2-medium",
    "--network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default",
    "--metadata=enable-osconfig=TRUE",
    "--maintenance-policy=MIGRATE",
    "--provisioning-model=STANDARD",
    "--service-account=1063515563667-compute@developer.gserviceaccount.com",
    "--scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append",
    "--create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054908,disk-resource-policy=projects/iceu-songpa07/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced",
    "--no-shielded-secure-boot",
    "--shielded-vtpm",
    "--shielded-integrity-monitoring",
    "--labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud",
    "--reservation-affinity=any",
    "--quiet"
]

print("[*] 인스턴스 생성 요청 시작...", flush=True)
subprocess.run(create_instance_cmd, shell=is_windows, stdin=subprocess.DEVNULL, check=True)
print("[+] 인스턴스 생성 완료!")

# 2. config.yaml 설정 파일 작성
config_yaml = """agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0
"""
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(config_yaml)
print("[+] config.yaml 생성 완료")

# 3. Ops Agent 정책 생성
ops_agent_cmd = [
    gcloud_bin, "compute", "instances", "ops-agents", "policies", "create",
    "goog-ops-agent-v2-template-1-7-0-us-central1-a",
    "--project=iceu-songpa07",
    "--zone=us-central1-a",
    "--file=config.yaml",
    "--quiet"
]
print("[*] Ops Agent 정책 생성 시작...", flush=True)
subprocess.run(ops_agent_cmd, shell=is_windows, stdin=subprocess.DEVNULL, check=True)
print("[+] 모든 작업이 성공적으로 완료되었습니다.")

[*] 인스턴스 생성 요청 시작...
[+] 인스턴스 생성 완료!
[+] config.yaml 생성 완료
[*] Ops Agent 정책 생성 시작...


CalledProcessError: Command '['C:\\Users\\admy7\\AppData\\Local\\Google\\Cloud SDK\\google-cloud-sdk\\bin\\gcloud.cmd', 'compute', 'instances', 'ops-agents', 'policies', 'create', 'goog-ops-agent-v2-template-1-7-0-us-central1-a', '--project=iceu-songpa07', '--zone=us-central1-a', '--file=config.yaml', '--quiet']' returned non-zero exit status 1.

### 방법 2. Bash 셀 매직(`%%bash`)으로 실행
Linux, macOS, Google Colab 또는 WSL/Git Bash 환경에서 직접 실행할 수 있는 원본 쉘 명령어입니다.

In [ ]:
%%bash
gcloud compute instances create instance-20260914-054908 \
    --project=iceu-songpa07 \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=1063515563667-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054908,disk-resource-policy=projects/iceu-songpa07/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any \
    --quiet \
&& \
printf 'agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n' > config.yaml \
&& \
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa07 \
    --zone=us-central1-a \
    --file=config.yaml \
    --quiet

### 방법 3. 🚨 잔여 리소스 및 과금 위험 점검 (Harness 실행)
실습 종료 전 또는 퇴근 전, 활성화된 VM, 디스크, 외부 IP 등 불필요하게 과금될 수 있는 리소스가 남아있는지 확인합니다.

In [ ]:
from check_gcp_resources import check_resources

# 잔여 리소스 및 과금 위험 점검 실행
check_resources()

### 방법 4. 작업 완료 후 인스턴스 및 정책 안전 정리 (과금 방지)
실습 완료 후 과금을 방지하기 위해 인스턴스와 Ops Agent 정책을 삭제합니다.

In [ ]:
import os
import sys
import shutil
import subprocess

gcloud_bin = shutil.which("gcloud.cmd") or shutil.which("gcloud") or "gcloud"
is_windows = sys.platform.startswith("win")

# 1. Ops Agent 정책 삭제
try:
    subprocess.run([
        gcloud_bin, "compute", "instances", "ops-agents", "policies", "delete",
        "goog-ops-agent-v2-template-1-7-0-us-central1-a",
        "--project=iceu-songpa07",
        "--zone=us-central1-a",
        "--quiet"
    ], shell=is_windows, stdin=subprocess.DEVNULL, check=True)
    print("[+] Ops Agent 정책 삭제 완료")
except subprocess.CalledProcessError as e:
    print("[-] 정책 삭제 건너뜀 또는 이미 삭제됨")

# 2. Compute Engine 인스턴스 삭제 (부팅 디스크도 함께 자동 삭제됨)
target_instances = ["instance-20260914-054908", "instance-cheapest-e2m"]
target_zone = "us-central1-a"

for inst_name in target_instances:
    try:
        subprocess.run([
            gcloud_bin, "compute", "instances", "delete", inst_name,
            "--project=iceu-songpa07",
            f"--zone={target_zone}",
            "--quiet"
        ], shell=is_windows, stdin=subprocess.DEVNULL, check=True)
        print(f"[+] {inst_name} 인스턴스 삭제 완료! (과금 차단)")
    except subprocess.CalledProcessError:
        print(f"[*] {inst_name} 인스턴스 미존재 (이미 삭제됨)")